# LF6 — degrade_boundary (controlled degradation) trên Google Colab

Sinh nhãn hữu dụng cho 5 tác vụ bằng **suy giảm có kiểm soát**: lấy ảnh **anchor**
(model hạ nguồn đọc đúng ground-truth ở delta=0, confidence > `TAU_HIGH`), suy giảm dần
theo từng trục cho tới khi tác vụ thất bại. Điểm chuyển 1->0 là **điểm gãy** `delta*`.
`delta < delta*` -> phiếu 1; `delta >= delta*` -> phiếu 0; `|delta - delta*| < eps` -> abstain.

Không rò rỉ: mỗi anchor thuộc fold `k` được chấm bằng model out-of-fold `M_{-k}`
(fold đọc thẳng từ `labels/votes/` — đã băm md5 theo ảnh gốc, khớp mọi LF).

Notebook clone repo về `/content/coconut-iqa`; repo phải đã commit `Dataset/`,
`src/utils/lf_io.ipynb`, và weights: `labels/lf1_yolov8/runs/fold{k}/weights/best.pt`
(maturity) + `labels/disease_clf/runs/fold{k}.pt` (bệnh, dùng chung task 2/3/4/5).
Xem `docs/LF6_Methodology.md`.

In [ ]:
%pip install -q ultralytics torchvision

In [ ]:
import platform
import subprocess
from pathlib import Path
import numpy as np
import pandas as pd

RUNNER = "colab"          # "colab" | "local" | "kaggle"
REPO_URL = "https://github.com/phuong-duong/coconut-iqa.git"


def ensure_clone(dest):
    if dest.exists():
        return
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(dest)],
        check=True,
    )


def resolve_repo(runner):
    if runner == "colab":
        repo = Path("/content/coconut-iqa")
        ensure_clone(repo)
        return repo
    if runner == "local":
        for cand in [Path.cwd(), *Path.cwd().parents]:
            if (cand / "Dataset").is_dir():
                return cand
        raise SystemExit("Khong thay repo coconut-iqa (Dataset/) — chay trong repo.")
    if runner == "kaggle":
        repo = Path("/kaggle/input/coconut-iqa")
        if not repo.exists():
            raise SystemExit("Khong thay /kaggle/input/coconut-iqa — them dataset repo.")
        return repo
    raise SystemExit("RUNNER phai la colab | local | kaggle")


REPO = resolve_repo(RUNNER)
VEIRF = REPO / "Dataset" / "coconut-veirf-v5"
DISEASE = REPO / "Dataset" / "Coconut Tree Disease Dataset"
VOTES_DIR = REPO / "labels" / "votes"
YOLO_RUN = REPO / "labels" / "lf1_yolov8" / "runs"
DISEASE_RUN = REPO / "labels" / "disease_clf" / "runs"
OUT_MANIFEST = REPO / "labels" / "lf6_degradation_manifest.csv"
UTILS = REPO / "src" / "utils" / "lf_io.ipynb"

for p in [VEIRF, DISEASE, VOTES_DIR, UTILS]:
    if not p.exists():
        raise SystemExit("Khong thay " + str(p) + " — repo da commit day du chua?")

# Module dung chung (TASKS, original_id, fold_of). %run vi la .ipynb.
get_ipython().run_line_magic("run", str(UTILS))
print("REPO:", REPO)

In [ ]:
import torch
import torchvision
import ultralytics

SEED = 42
K = 5
TAU_HIGH = 0.85           # cong chon anchor (gold report LF1 goi y tau=0.85)
EPS_FRAC = 0.10           # cong bien abstain: |delta - delta*| < EPS_FRAC * span
YOLO_CONF = 0.25          # nguong conf doc box YOLO (khop CONF_TAU cua LF1)
IOU_THR = 0.5             # nguong IoU ghep cap box (khop LF1)
IMGSZ_YOLO = 640
IMGSZ_DISEASE = 224

STAGE = {0: "dry", 1: "green", 2: "tender"}
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

DISEASE_KEYS = ["gls", "leafrot", "stembleed", "budrot", "budroot"]
FOLDER_KEY = {}
FOLDER_KEY["Gray Leaf Spot"] = "gls"
FOLDER_KEY["Leaf Rot"] = "leafrot"
FOLDER_KEY["Stem Bleeding"] = "stembleed"
FOLDER_KEY["Bud Rot"] = "budrot"
FOLDER_KEY["Bud Root Dropping"] = "budroot"

VOTE_FILE = {}
VOTE_FILE["1_maturity_evaluation"] = "lf1_maturity.csv"
VOTE_FILE["2_foliar_disease"] = "lf2_foliar.csv"
VOTE_FILE["3_trunk_disease"] = "lf3_trunk.csv"
VOTE_FILE["4_crown_disease"] = "lf4_crown.csv"
VOTE_FILE["5_petiole"] = "lf5_petiole.csv"

ALL_AXES = ["blur", "exposure", "resolution", "occlusion", "white_balance"]
AXES_BY_TASK = {}
AXES_BY_TASK["1_maturity_evaluation"] = ["blur", "resolution", "occlusion"]
AXES_BY_TASK["2_foliar_disease"] = ALL_AXES
AXES_BY_TASK["3_trunk_disease"] = ALL_AXES
AXES_BY_TASK["4_crown_disease"] = ALL_AXES
AXES_BY_TASK["5_petiole"] = ALL_AXES

GRIDS = {}
GRIDS["blur"] = [0.0, 1.0, 2.0, 3.0, 5.0, 7.0, 10.0]
GRIDS["exposure"] = [0.0, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0]
GRIDS["resolution"] = [0.0, 0.2, 0.4, 0.6, 0.75, 0.85, 0.9]
GRIDS["occlusion"] = [0.0, 0.1, 0.2, 0.3, 0.45, 0.6, 0.75]
GRIDS["white_balance"] = [0.0, 0.1, 0.2, 0.3, 0.45, 0.6, 0.8]

DEVICE = "cpu"
GPU_NAME = "CPU"
if torch.cuda.is_available():
    DEVICE = "cuda"
    GPU_NAME = torch.cuda.get_device_name(0)
elif torch.backends.mps.is_available():
    DEVICE = "mps"
    GPU_NAME = "mps"

print("python:", platform.python_version())
print("torch:", torch.__version__, "| torchvision:", torchvision.__version__, "| ultralytics:", ultralytics.__version__)
print("device:", DEVICE, "| gpu:", GPU_NAME)
print("SEED:", SEED, "| K:", K, "| TAU_HIGH:", TAU_HIGH, "| EPS_FRAC:", EPS_FRAC)
print("YOLO_CONF:", YOLO_CONF, "| IOU_THR:", IOU_THR)
for axis in ALL_AXES:
    print("grid", axis, GRIDS[axis])
for task in TASKS:
    print("axes", task, AXES_BY_TASK[task])

## Toán tử suy giảm

Mỗi toán tử: `D(img, 0.0) == img` và đơn điệu theo `delta`. Độ chín chỉ dùng
`blur/resolution/occlusion` (ảnh Roboflow đã bị auto-contrast + phơi sáng ×3).

In [ ]:
import cv2
from PIL import Image


def _to_arr(img):
    return np.asarray(img.convert("RGB"), dtype=np.float32)


def _to_img(arr):
    clipped = np.clip(arr, 0, 255).astype(np.uint8)
    return Image.fromarray(clipped, "RGB")


def deg_blur(img, delta):
    if delta <= 0:
        return img
    arr = _to_arr(img)
    k = int(round(delta)) * 2 + 1
    out = cv2.GaussianBlur(
        src=arr,
        ksize=(k, k),
        sigmaX=float(delta),
    )
    return _to_img(out)


def deg_exposure(img, delta):
    if delta <= 0:
        return img
    factor = 2.0 ** (-delta)
    return _to_img(_to_arr(img) * factor)


def deg_resolution(img, delta):
    if delta <= 0:
        return img
    w, h = img.size
    f = max(0.02, 1.0 - float(delta))
    small_w = max(1, int(w * f))
    small_h = max(1, int(h * f))
    small = img.resize((small_w, small_h), Image.BILINEAR)
    return small.resize((w, h), Image.BILINEAR).convert("RGB")


def deg_occlusion(img, delta):
    if delta <= 0:
        return img
    arr = _to_arr(img)
    h = arr.shape[0]
    w = arr.shape[1]
    frac = min(0.95, float(delta))
    bh = int(round(h * frac ** 0.5))
    bw = int(round(w * frac ** 0.5))
    y0 = (h - bh) // 2
    x0 = (w - bw) // 2
    arr[y0:y0 + bh, x0:x0 + bw, :] = 0.0
    return _to_img(arr)


def deg_white_balance(img, delta):
    if delta <= 0:
        return img
    arr = _to_arr(img)
    arr[..., 0] *= (1.0 + float(delta))
    arr[..., 2] *= (1.0 - 0.5 * float(delta))
    return _to_img(arr)


OPERATORS = {}
OPERATORS["blur"] = deg_blur
OPERATORS["exposure"] = deg_exposure
OPERATORS["resolution"] = deg_resolution
OPERATORS["occlusion"] = deg_occlusion
OPERATORS["white_balance"] = deg_white_balance


def degrade(img, axis, delta):
    return OPERATORS[axis](img, delta)

## Điểm gãy trên bao đơn điệu + phiếu λ

In [ ]:
def break_point(success_seq, grid):
    # delta* tren bao don dieu: s_bar = cummin(success). Bo qua diem None.
    s_bar = float("inf")
    for s, d in zip(success_seq, grid):
        if s is None:
            continue
        s_bar = min(s_bar, s)
        if s_bar == 0:
            return float(d)
    return float("inf")


def lf6_vote(delta, delta_star, eps):
    if abs(delta - delta_star) < eps:
        return None
    if delta < delta_star:
        return 1
    return 0

## Self-test (logic thuần, không cần model)

In [ ]:
def _selftest():
    rng = np.random.default_rng(0)
    img = _to_img(rng.integers(0, 256, size=(64, 64, 3)).astype(np.float32))
    a0 = _to_arr(img)
    for axis in ALL_AXES:
        assert np.array_equal(_to_arr(degrade(img, axis, 0.0)), a0), axis
        diffs = []
        for d in GRIDS[axis]:
            diffs.append(float(np.abs(_to_arr(degrade(img, axis, d)) - a0).mean()))
        for i in range(len(diffs) - 1):
            assert diffs[i] <= diffs[i + 1] + 1e-6, (axis, diffs)
    grid = GRIDS["blur"]
    assert break_point([1, 1, 1, 0, 1, 0, 0], grid) == 3.0
    assert break_point([1, 1, 1, 1, 1, 1, 1], grid) == float("inf")
    assert break_point([1, 1, None, 0, 0, 0, 0], grid) == 3.0
    assert lf6_vote(2.0, 5.0, 1.0) == 1
    assert lf6_vote(7.0, 5.0, 1.0) == 0
    assert lf6_vote(5.0, 5.0, 1.0) is None
    print("SELF-TEST OK")


_selftest()

## Ground-truth + chọn anchor từ `labels/votes/`

In [ ]:
def as_int(x):
    try:
        return int(float(x))
    except (TypeError, ValueError):
        return -1


def as_float(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return 0.0


def read_boxes(txt):
    out = []
    if not txt.exists():
        return out
    for line in txt.read_text().splitlines():
        parts = line.split()
        if len(parts) < 5:
            continue
        cls = int(parts[0])
        box = (float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4]))
        out.append((cls, box))
    return out


def maturity_image_gt(rel_path):
    # rel_path vd 'train/images/010_....jpg' (tuong doi VEIRF)
    img = VEIRF / rel_path
    txt = Path(str(img).replace("/images/", "/labels/")).with_suffix(".txt")
    return img, read_boxes(txt)


def disease_image_key(rel_path):
    # path co the la 'Coconut Tree Disease Dataset/<folder>/<file>' hoac co tien to 'Dataset/'
    parts = Path(rel_path).parts
    marker = "Coconut Tree Disease Dataset"
    if marker in parts:
        idx = parts.index(marker)
        sub = Path(*parts[idx + 1:])
    else:
        sub = Path(rel_path)
    folder = sub.parts[0]
    img = DISEASE / sub
    key = FOLDER_KEY.get(folder)
    return img, key


def load_anchors():
    anchors = []
    for task in TASKS:
        vf = VOTES_DIR / VOTE_FILE[task]
        if not vf.exists():
            raise SystemExit("Khong thay " + str(vf))
        d = pd.read_csv(vf)
        if "fold" not in d.columns or "confidence" not in d.columns:
            print("bo qua", task, "(file phieu thieu cot fold/confidence)")
            continue
        for r in d.itertuples():
            if as_int(r.vote) != 1:
                continue
            if as_float(r.confidence) <= TAU_HIGH:
                continue
            fold = as_int(r.fold)
            if fold < 0:
                continue
            if task == "1_maturity_evaluation":
                img, boxes = maturity_image_gt(r.path)
                if not boxes:
                    continue
                anchors.append(dict(image_id=r.image_id, source=r.source, task=task, fold=fold, img=img, kind="maturity", gt=boxes))
            else:
                img, key = disease_image_key(r.path)
                if key is None:
                    continue
                anchors.append(dict(image_id=r.image_id, source=r.source, task=task, fold=fold, img=img, kind="disease", gt=key))
    return anchors


anchors = load_anchors()
print("anchor:", len(anchors))
by_task = {}
for a in anchors:
    by_task[a["task"]] = by_task.get(a["task"], 0) + 1
for task in TASKS:
    print("  ", task, by_task.get(task, 0))

## Nạp model out-of-fold (YOLOv8 maturity + disease classifier dùng chung)

In [ ]:
import torch.nn as nn
from ultralytics import YOLO
from torchvision import models
from torchvision import transforms


def load_yolo_folds():
    out = {}
    for k in range(K):
        w = YOLO_RUN / ("fold" + str(k)) / "weights" / "best.pt"
        if not w.exists():
            raise SystemExit("Thieu YOLO weights: " + str(w))
        out[k] = YOLO(str(w))
    return out


def build_disease_model():
    model = models.efficientnet_b0(weights=None)
    in_features = model.classifier[-1].in_features
    model.classifier[-1] = nn.Linear(in_features, len(DISEASE_KEYS))
    return model


def load_disease_folds():
    out = {}
    for k in range(K):
        w = DISEASE_RUN / ("fold" + str(k) + ".pt")
        if not w.exists():
            raise SystemExit("Thieu disease weights: " + str(w))
        model = build_disease_model()
        state = torch.load(str(w), map_location=DEVICE)
        model.load_state_dict(state)
        model.eval()
        model.to(DEVICE)
        out[k] = model
    return out


disease_eval_tf = transforms.Compose([
    transforms.Resize(IMGSZ_DISEASE + 32),
    transforms.CenterCrop(IMGSZ_DISEASE),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

YOLO_FOLDS = load_yolo_folds()
DISEASE_FOLDS = load_disease_folds()
print("YOLO folds:", len(YOLO_FOLDS), "| disease folds:", len(DISEASE_FOLDS))

## Hàm `success` out-of-fold (tái dùng đúng logic correctness của LF1)

In [ ]:
def iou(a, b):
    ax1 = a[0] - a[2] / 2
    ay1 = a[1] - a[3] / 2
    ax2 = a[0] + a[2] / 2
    ay2 = a[1] + a[3] / 2
    bx1 = b[0] - b[2] / 2
    by1 = b[1] - b[3] / 2
    bx2 = b[0] + b[2] / 2
    by2 = b[1] + b[3] / 2
    iw = max(0.0, min(ax2, bx2) - max(ax1, bx1))
    ih = max(0.0, min(ay2, by2) - max(ay1, by1))
    inter = iw * ih
    union = a[2] * a[3] + b[2] * b[3] - inter
    if union <= 0:
        return 0.0
    return inter / union


def lf1_vote(gt_boxes, pred_boxes):
    # Tra 1/0/None (None = abstain). Khop dung logic notebook LF1.
    if not gt_boxes:
        return None
    confident = []
    for p in pred_boxes:
        if p[1] >= YOLO_CONF:
            confident.append(p)
    if not confident:
        return None
    n_correct = 0
    n_wrong = 0
    for pcls, pconf, pbox in confident:
        ious = []
        for gcls, gbox in gt_boxes:
            ious.append(iou(pbox, gbox))
        j = int(np.argmax(ious))
        if ious[j] < IOU_THR:
            continue
        if pcls == gt_boxes[j][0]:
            n_correct = n_correct + 1
        else:
            n_wrong = n_wrong + 1
    if n_correct >= 1 and n_wrong == 0:
        return 1
    return 0


def yolo_pred_boxes(model, pil_img):
    results = model.predict(
        source=pil_img,
        conf=0.01,
        imgsz=IMGSZ_YOLO,
        device=DEVICE,
        verbose=False,
    )
    boxes = []
    for out in results:
        for b in out.boxes:
            cls = int(b.cls[0])
            conf = float(b.conf[0])
            xywh = tuple(map(float, b.xywhn[0]))
            boxes.append((cls, conf, xywh))
    return boxes


def maturity_success(gt_boxes, pil_img, fold):
    model = YOLO_FOLDS[fold]
    preds = yolo_pred_boxes(model, pil_img)
    return lf1_vote(gt_boxes, preds)


def disease_success(gt_key, pil_img, fold):
    model = DISEASE_FOLDS[fold]
    x = disease_eval_tf(pil_img.convert("RGB")).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logits = model(x)
        probs = torch.softmax(logits, dim=1)[0]
    best = int(torch.argmax(probs))
    if DISEASE_KEYS[best] == gt_key:
        return 1
    return 0

## Chạy: dò điểm gãy từng anchor × trục, ghi manifest (append + resume)

In [ ]:
import csv

MANIFEST_FIELDS = ["base_image", "original_id", "fold", "source", "task", "axis", "delta", "delta_star", "vote"]


def manifest_done_keys(path):
    keys = set()
    if not Path(path).exists():
        return keys
    d = pd.read_csv(path)
    for r in d.itertuples():
        keys.add((str(r.base_image), str(r.axis)))
    return keys


def append_rows(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    is_new = not path.exists()
    handle = path.open("a", newline="")
    writer = csv.DictWriter(handle, fieldnames=MANIFEST_FIELDS)
    if is_new:
        writer.writeheader()
    for r in rows:
        writer.writerow(r)
    handle.flush()
    handle.close()


done = manifest_done_keys(OUT_MANIFEST)
print("da co", len(done), "cap (base_image, axis) — resume")
n_written = 0
for a in anchors:
    oid = original_id(a["image_id"], a["source"])
    base = Image.open(a["img"]).convert("RGB")
    for axis in AXES_BY_TASK[a["task"]]:
        if (a["image_id"], axis) in done:
            continue
        grid = GRIDS[axis]
        span = grid[-1] - grid[0]
        eps = EPS_FRAC * span
        seq = []
        for delta in grid:
            dimg = degrade(base, axis, delta)
            if a["kind"] == "maturity":
                s = maturity_success(a["gt"], dimg, a["fold"])
            else:
                s = disease_success(a["gt"], dimg, a["fold"])
            seq.append(s)
        dstar = break_point(seq, grid)
        rows = []
        for delta in grid:
            v = lf6_vote(delta, dstar, eps)
            if v is None:
                vout = ""
            else:
                vout = v
            row = {}
            row["base_image"] = a["image_id"]
            row["original_id"] = oid
            row["fold"] = a["fold"]
            row["source"] = a["source"]
            row["task"] = a["task"]
            row["axis"] = axis
            row["delta"] = delta
            row["delta_star"] = dstar
            row["vote"] = vout
            rows.append(row)
        append_rows(OUT_MANIFEST, rows)
        n_written = n_written + len(rows)
        print(a["image_id"], a["task"], axis, "delta*", dstar, flush=True)
print("ghi them", n_written, "dong ->", OUT_MANIFEST, flush=True)

In [ ]:
m = pd.read_csv(OUT_MANIFEST)
print("tong dong:", len(m))
for t in TASKS:
    sub = m[m.task == t]
    if len(sub) == 0:
        continue
    pos = int((sub.vote == 1).sum())
    neg = int((sub.vote == 0).sum())
    ab = int(sub.vote.isna().sum())
    print(t, "1", pos, "0", neg, "abstain", ab)

## Tải manifest về (kẻo mất khi phiên Colab đóng)

In [ ]:
# Manifest ghi trong repo clone (/content/...) -> phien Colab dong la mat.
# Tai ve; hoac commit + push neu muon giu trong git.
if RUNNER == "colab":
    from google.colab import files
    files.download(str(OUT_MANIFEST))
else:
    print("manifest:", OUT_MANIFEST)